# Question 2: When clustering 5-man lineups by offensive and defensive efficiency (OffRtg, DefRtg, TS%, eFG%, Pace), what distinct lineup “styles” emerge, and which player archetype combinations dominate each cluster?


In [ ]:
# 1. Imports

import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from collections import Counter

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Folders (adjust if needed)
LINEUP_DIR    = "NBA5ManLineupStats"
ARCHETYPE_DIR = "NBAPlayerArchetypesAssigned"

In [ ]:
# 2. Helper Functions

def season_from_filename(fname: str) -> str:
    """
    Extract season key from filenames like 
    'NBA5ManLineupStats(2020-21).csv' -> '2020-21'
    """
    m = re.search(r"\((\d{4}-\d{2})\)", fname)
    if not m:
        raise ValueError(f"Cannot parse season from filename: {fname}")
    return m.group(1)


def parse_lineup_players(lineup_str):
    """
    Split lineup string 'A. Player - B. Player - ...' into list of names.
    Handles NaN, None, floats, and unexpected formats gracefully.
    """
    if not isinstance(lineup_str, str):
        return []   # return empty list for bad rows
    
    if "-" not in lineup_str:
        return []   # malformed lineup
    
    return [p.strip() for p in lineup_str.split(" - ") if isinstance(p, str)]

In [ ]:
# 3. Load Archetypes For All Seasons and Match Names

archetypes_by_season = {}

for fname in os.listdir(ARCHETYPE_DIR):
    if not fname.endswith(".csv"):
        continue
    
    season = season_from_filename(fname)
    path = os.path.join(ARCHETYPE_DIR, fname)
    df = pd.read_csv(path)
    
    # basic cleaning
    df["Player"]    = df["Player"].str.strip()
    df["Team"]      = df["Team"].str.strip()
    df["Archetype"] = df["Archetype"].str.strip()
    
    archetypes_by_season[season] = df

print("Loaded archetypes for seasons:", sorted(archetypes_by_season.keys()))

def match_player_name(initial_last: str, team: str, arch_df: pd.DataFrame):
    """
    Match lineup names like 'S. Curry' to a full name in the archetype file.
    Uses:
      - Last name match
      - First initial match
      - Team filter when possible
    """
    if pd.isna(initial_last):
        return None
    
    name = initial_last.strip()

    # Handle common lineup formats
    if "." in name:
        parts = name.split(".")
        init = parts[0].strip().lower()
        last = parts[1].strip().lower()
    else:
        # fallback: try last name only
        init = None
        last = name.lower()

    # Filter by last name containing match
    candidates = arch_df[arch_df["Player"].str.lower().str.contains(last)]

    if candidates.empty:
        return None

    # Team filter (helps disambiguate Lopez, Williams, Brown, etc.)
    if "Team" in arch_df.columns:
        team_matches = candidates[candidates["Team"].str.lower() == team.lower()]
        if len(team_matches) > 0:
            candidates = team_matches

    # Initial filter
    if init is not None:
        initial_matches = [
            row for _, row in candidates.iterrows()
            if row["Player"].lower().strip().startswith(init)
        ]
        if len(initial_matches) > 0:
            return initial_matches[0]["Player"]

    # Fallback: closest-looking match (first)
    return candidates.iloc[0]["Player"]

 

In [ ]:
# 4. Map Archetypes

ROLE_GROUPS = ["Shooter", "Playmaker", "Defensive", "Scorer", "Versatile", "Other"]

def map_archetype_to_role(arch: str) -> str:
    """
    Map a fine-grained archetype into a coarse role group.
    Uses exact dictionary matching so category assignment is reliable.
    """
    if pd.isna(arch):
        return "Other"

    # Normalize key
    key = arch.strip()

    # Master mapping (based on your definition table)
    role_map = {
        "Stretch Big":              "Shooter",
        "Rim running shotblocker":  "Defensive",
        "Interior Creator":         "Scorer",

        "Combo Guard":              "Scorer",
        "Lead Playmaker":           "Playmaker",
        "Secondary Playmaker":      "Playmaker",

        "Pitbull":                  "Defensive",
        "3&D Specialist":           "Shooter",
        "2 Way Player":             "Versatile",

        "Slasher":                  "Scorer",
        "Point Forward":            "Playmaker",
        "Shooter":                  "Shooter",
        "Switchable defender":      "Defensive",
        "Playmaking forward":       "Playmaker",

        "Microwave":                "Scorer",
        "Glue guy":                 "Versatile",
        "Athletic Finisher":        "Scorer",
    }

    # Return matched role, or Other if not recognized
    return role_map.get(key, "Other")

def get_player_archetype(player_name: str, team: str, season: str) -> str:
    """
    Look up archetype for (player, team, season). 
    Falls back to player-only match, then returns NaN if not found.
    """
    arch_df = archetypes_by_season.get(season)
    if arch_df is None:
        return np.nan
    
    # exact player+team match
    mask = (arch_df["Player"] == player_name)
    if team is not None and "Team" in arch_df.columns:
        mask &= (arch_df["Team"] == team)
    
    subset = arch_df[mask]
    if not subset.empty:
        return subset.iloc[0]["Archetype"]
    
    # fallback: player only
    subset = arch_df[arch_df["Player"] == player_name]
    if not subset.empty:
        return subset.iloc[0]["Archetype"]
    
    return np.nan

In [ ]:
# 5. Lineup dataframe

lineup_rows = []

for fname in os.listdir(LINEUP_DIR):
    if not fname.endswith(".csv"):
        continue
    
    season = season_from_filename(fname)
    print(f"Processing season {season}...")
    
    path = os.path.join(LINEUP_DIR, fname)
    df_lineups = pd.read_csv(path)
    
    # Basic cleanliness
    df_lineups["Lineups"] = df_lineups["Lineups"].str.strip()
    df_lineups["TEAM"]    = df_lineups["TEAM"].str.strip()
    
    # Clean lineup strings before processing
    df_lineups["Lineups"] = df_lineups["Lineups"].astype(str).str.strip()

    # Remove rows where lineup string is "nan" or empty
    df_lineups = df_lineups[df_lineups["Lineups"].str.contains(" - ", na=False)]
    
    for _, row in df_lineups.iterrows():
        lineup_str = row["Lineups"]
        team       = row["TEAM"]
        
        players = parse_lineup_players(lineup_str)

        arch_df = archetypes_by_season[season]   # get correct archetype file for season
        roles = []

        for p in players:
            # Convert "S. Curry" → "Stephen Curry"
            full_name = match_player_name(p, team, arch_df)

            if full_name in arch_df["Player"].values:
                fine_arch = arch_df.loc[
                    arch_df["Player"] == full_name, "Archetype"
                ].iloc[0]
            else:
                fine_arch = None

            # map to role group (Shooter, Creator, etc.)
            role = map_archetype_to_role(fine_arch)
            roles.append(role)

        role_counts = Counter(roles)
        
        # build row dict
        row_dict = {
            "Season": season,
            "TEAM":   team,
            "Lineups": lineup_str,
            "GP":   row.get("GP", np.nan),
            "MIN":  row.get("MIN", np.nan),
            "OffRtg": row.get("OffRtg", np.nan),
            "DefRtg": row.get("DefRtg", np.nan),
            "NetRtg": row.get("NetRtg", np.nan),
            "TS%":    row.get("TS%", np.nan),
            "eFG%":   row.get("eFG%", np.nan),
            "PACE":   row.get("PACE", np.nan),
            "PIE":    row.get("PIE", np.nan),
            "REB%":    row.get("REB%", np.nan),
            "TO Ratio":    row.get("TO Ratio", np.nan),
        }
        
        # add role-group counts
        for g in ROLE_GROUPS:
            row_dict[f"role_{g}"] = role_counts.get(g, 0)
        
        lineup_rows.append(row_dict)

lineup_df = pd.DataFrame(lineup_rows)
print("Lineup_df shape:", lineup_df.shape)


In [ ]:
# 6. Role Groups Counts

stat_cols = ["OffRtg", "DefRtg", "TS%", "eFG%", "PACE", "REB%", "TO Ratio"]
role_cols = [c for c in lineup_df.columns if c.startswith("role_")]

print("Stat columns:", stat_cols)
print("Role-group columns:", role_cols)

X_stats = lineup_df[stat_cols].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_stats)

In [ ]:
# 7. Silhouette Score to choose K

k_values = range(2, 9)  # try 3–8 clusters
sil_scores = []

for k in k_values:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=20)
    labels = kmeans.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    sil_scores.append(score)
    print(f"k={k}: silhouette={score:.3f}")

plt.figure(figsize=(5, 4))
plt.plot(list(k_values), sil_scores, marker="o")
plt.xlabel("k (number of clusters)")
plt.ylabel("Silhouette score")
plt.title("Silhouette score vs k")
plt.grid(True)
plt.show()

In [ ]:
# 8. Fit final K-Means model & attach cluster labels

FINAL_K = 5  # <-- adjust after checking silhouette plot

kmeans = KMeans(n_clusters=FINAL_K, random_state=42, n_init=20)
cluster_labels = kmeans.fit_predict(X_scaled)

lineup_df["Cluster"] = cluster_labels
lineup_df["ClusterLabel"] = lineup_df["Cluster"].map(lambda x: f"Cluster {x}")
lineup_df["Cluster"].value_counts().sort_index()

In [ ]:
# 9. Scaled cluster lineup stats heatmap (standardized across metrics)

cluster_stats = (
    lineup_df
    .groupby("Cluster")[stat_cols + ["NetRtg"]]
    .mean()
)

# Z-score (column-wise scaling)
cluster_stats_scaled = (cluster_stats - cluster_stats.mean()) / cluster_stats.std()

# Round for readability
cluster_stats_scaled = cluster_stats_scaled.round(2)

print("Scaled (z-score) stats by cluster:")
display(cluster_stats_scaled)

plt.figure(figsize=(7, 4))
sns.heatmap(cluster_stats_scaled.T, annot=True, cmap="coolwarm", center=0)
plt.title("Cluster Averages (Standardized)")
plt.ylabel("Metric")
plt.xlabel("Cluster")
plt.tight_layout()
plt.show()

In [ ]:
# 10. Archetype dist by cluster

cluster_roles = (
    lineup_df
    .groupby("Cluster")[role_cols]
    .mean()
    .round(2)
)

print("Average role-group counts per lineup by cluster:")
display(cluster_roles)

num_clusters = lineup_df["Cluster"].nunique()
fig, axes = plt.subplots(1, num_clusters, figsize=(4*num_clusters, 4), sharey=True)

if num_clusters == 1:
    axes = [axes]

for c in range(num_clusters):
    ax = axes[c]
    vals = cluster_roles.loc[c]
    vals.plot(kind="bar", ax=ax)
    ax.set_title(f"Cluster {c}")
    ax.set_xticklabels(vals.index, rotation=45, ha="right")
    ax.set_ylabel("Avg count per lineup")

plt.suptitle("Role-Group Composition by Cluster", y=1.02, fontsize=14)
plt.tight_layout()
plt.show()